# Job Scraper for Google Colab (v2.0)
## Scrape Multiple Job Categories with Multilingual Support

This notebook scrapes job listings from LinkedIn with advanced filtering for language requirements and working student positions.

**Features:**
- Support for multiple job types (AIjobs, PMjobs, Datajobs, Autojobs, ResAsstjobs, VLSIjobs, ECEjobs)
- Multilingual filtering (German DE, Italian IT)
- Google Drive integration for saving results to your personal storage
- Smart employment type filtering (working student vs. full-time)

**Run all cells from top to bottom in order.**

## 1. Setup Google Colab Environment

In [ ]:
import sys
print(f"Python Version: {sys.version}")

# Check if running in Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except:
    IN_COLAB = False
    print("Running in Local Environment")

## 2. Install Required Dependencies

In [ ]:
import subprocess
import os

# Install required packages
packages = ['numpy','python-jobspy', 'pandas']
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("All dependencies installed successfully!")

## 3. Import Libraries

In [ ]:
from jobspy import scrape_jobs
import pandas as pd
from datetime import datetime
import re
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 4. Configure Job Classes and Filtering Keywords

In [ ]:
class JobClass:
    """Base class for job categories."""
    
    def __init__(self, name, search_terms):
        self.name = name
        self.search_terms = search_terms
    
    def __repr__(self):
        return f"{self.name}({len(self.search_terms)} search terms)"


class PMjobs(JobClass):
    """Product/Project Management related jobs."""
    
    def __init__(self):
        search_terms = [
            "Project Manager", "Project Leader", "Project Management",
            "Project Lead", "Product Manager"
        ]
        super().__init__("PMjobs", search_terms)


class Datajobs(JobClass):
    """Data Engineering & Analytics related jobs."""
    
    def __init__(self):
        search_terms = ["Data Engineering", "Data Analytics"]
        super().__init__("Datajobs", search_terms)


class AIjobs(JobClass):
    """AI/ML related jobs."""
    
    def __init__(self):
        search_terms = [
            "machine learning", "artificial intelligence",
            "Generative AI", "data science", "data scientist"
        ]
        super().__init__("AIjobs", search_terms)

class Autojobs(JobClass):
    """Automotive Engineering related jobs."""
    
    def __init__(self):
        search_terms = [
            "Automotive Design Engineer", "Vehicle Dynamics Engineer",
            "Automotive Systems Engineer", "CAD Design Engineer (Automotive)",
            "Powertrain Engineer"
        ]
        super().__init__("Autojobs", search_terms)


class ResAsstjobs(JobClass):
    """Research Assistant & Graduate Research positions."""
    
    def __init__(self):
        search_terms = ["Research Assistant", "Graduate Research"]
        super().__init__("ResAsstjobs", search_terms)


class VLSIjobs(JobClass):
    """VLSI/Chip Design Engineering related jobs."""
    
    def __init__(self):
        search_terms = [
            "chip design", "FPGA Engineer", "RTL Design",
            "ASIC Design Engineer", "FPGA Developer"
        ]
        super().__init__("VLSIjobs", search_terms)


class ECEjobs(JobClass):
    """Electronics & Communications Engineering related jobs."""
    
    def __init__(self):
        search_terms = [
            "Embedded Systems Engineer", "Hardware Design Engineer",
            "Signal Processing Engineer", "RF Engineer", "PCB Design Engineer"
        ]
        super().__init__("ECEjobs", search_terms)


# Registry of available job classes
JOB_CLASSES = {
    "PMjobs": PMjobs,
    "Datajobs": Datajobs,
    "AIjobs": AIjobs,
    "Autojobs": Autojobs,
    "ResAsstjobs": ResAsstjobs,
    "VLSIjobs": VLSIjobs,
    "ECEjobs": ECEjobs
}


def get_job_class(job_type):
    """Get a job class instance by name."""
    if job_type not in JOB_CLASSES:
        available = ", ".join(sorted(JOB_CLASSES.keys()))
        raise ValueError(f"Unknown job type '{job_type}'. Available types: {available}")
    return JOB_CLASSES[job_type]()

In [ ]:
# Configuration
SITES = ["linkedin", "google"]
RESULTS_WANTED = 300
HOURS_OLD = 168  # Last 7 days

# German language requirement keywords
GERMAN_REQUIRED_KEYWORDS = [
    "C2 Deutsch", "C1 Deutsch",
    "Deutschkenntnisse auf C2-Niveau", "Deutschkenntnisse auf C1-Niveau",
    "both in German (C2) and", "both in German (C1) and",
    "Deutsch - Fließend",
    "Sehr gute Kommunikationsfähigkeiten in deutscher und",
    "verhandlungssicher Deutsch", "verhandlungssichere Deutschkenntnisse",
    "sehr gute Deutschkenntnisse", "Sehr gute Deutsch",
    "fließend Deutsch",
    "fluent in German", "Fluent German",
    "kommunizierst sicher auf Deutsch",
    "Deutschkenntnisse sind verhandlungssicher",
    "business fluent German",
    "Deutschkenntnisse auf Muttersprachenniveau",
    "Deutsch und Englisch verhandlungssicher in Wort und Schrift",
    "sehr gute Deutsch- und Englischkenntnisse in Wort und Schrift",
    "Präsentationsfähigkeiten in Deutsch-",
    "Präsentationsfähigkeiten in Deutsch",
    "Beratungsfähigkeiten in Deutsch-",
    "Beratungsfähigkeiten in Deutsch",
    "Kommunikationsfähigkeiten in Deutsch",
    "Fluent in English and German",
    "mit Kunden / Kundenkontakt",
    "Präsentationen beim Kunden",
    "Stakeholder bis auf C-Level",
    "Excellent German",
    "Excellent English and German",
    "proficiency in German",
    "proficiency in English and German",
    "Very good English and German" 
]

# Working student keywords
STUDENT_KEYWORDS = [
    "werkstudent", "intern", "thesis",
    "working student", "student assistant", "student worker",
    "praktikum", "hiwi", "abschlussarbeit",
    "internship", "werkstudenten"
]

# Search location (country-wide by default, filtered by CUSTOM_LOCATIONS if provided)
SEARCH_LOCATION = "Global"  # Can customize based on language preference

print("Configuration loaded!")

## 5. Define Helper Functions

In [ ]:
def filter_by_location(df, custom_locations=None, is_working_student=False):
    """
    Filter jobs by location (optional).
    
    Args:
        df: DataFrame with job listings
        custom_locations: Optional list of location strings to filter by
        is_working_student: Boolean to include remote jobs with location filter
        
    Returns:
        Filtered DataFrame (original if no custom_locations provided)
    """
    if not custom_locations or len(custom_locations) == 0:
        log_time("No location filter applied (location-agnostic search)")
        return df
    
    log_time(f"Filtering for location: {', '.join(custom_locations)}")
    escaped_locations = [re.escape(loc) for loc in custom_locations]
    location_pattern = "|".join(escaped_locations)
    location_match = df["location"].str.contains(location_pattern, case=False, na=False, regex=True)
    
    if is_working_student:
        log_time(f"Including remote jobs for working student positions...")
        is_remote = df["is_remote"].fillna(False) == True
        df = df[location_match | is_remote]
    else:
        df = df[location_match]
    
    return df

In [ ]:
def process_and_save_results(job_class, include_ws, include_nonws):
    """Main processing function."""
    
    log_time(f"=== Starting Job Scraper for {job_class.name} ===")
    log_time(f"Search terms: {', '.join(job_class.search_terms[:3])}{'...' if len(job_class.search_terms) > 3 else ''}")
    log_time(f"Include working student: {include_ws}")
    log_time(f"Include non-working student: {include_nonws}")
    
    # Scrape jobs
    all_jobs = scrape_jobs_for_terms(job_class.search_terms, GERMANY_SEARCH_LOCATION)
    
    if not all_jobs:
        log_time("No jobs found. Exiting.")
        return None, None
    
    log_time("\n=== Processing Results ===")
    log_time("Concatenating results...")
    df = pd.concat(all_jobs, ignore_index=True)
    
    # Remove duplicates
    log_time("Removing duplicates...")
    df_before = len(df)
    df = df.drop_duplicates(subset=["title", "company", "location"])
    log_time(f"Removed {df_before - len(df)} duplicates")
    log_time(f"Total jobs scraped: {len(df)}")
    
    # Filter out German language requirements
    df = filter_german_requirements(df)
    
    # Separate working student jobs from non-working student jobs
    log_time("Separating working student jobs from other jobs...")
    escaped_keywords = [re.escape(kw) for kw in STUDENT_KEYWORDS]
    student_pattern = "|".join(escaped_keywords)
    student_job_mask = df["title"].str.contains(student_pattern, case=False, na=False, regex=True)
    df_working_student = df[student_job_mask]
    df_non_working_student = df[~student_job_mask]
    
    log_time(f"Working student jobs: {len(df_working_student)}")
    log_time(f"Non-working student jobs: {len(df_non_working_student)}")
    
    # Apply location filtering
    if include_ws and len(df_working_student) > 0:
        df_working_student = filter_by_location(df_working_student, is_working_student=True)
        log_time(f"Working student jobs after location filter: {len(df_working_student)}")
    
    if include_nonws and len(df_non_working_student) > 0:
        df_non_working_student = filter_by_location(df_non_working_student, is_working_student=False)
        log_time(f"Non-working student jobs after location filter: {len(df_non_working_student)}")
    
    log_time("\n=== Processing Complete ===")
    
    # Prepare dataframes for saving
    df_ws_save = prepare_dataframe_for_save(df_working_student.copy()) if len(df_working_student) > 0 else df_working_student
    df_nonws_save = prepare_dataframe_for_save(df_non_working_student.copy()) if len(df_non_working_student) > 0 else df_non_working_student
    
    return df_ws_save, df_nonws_save

## 6. Mount Google Drive (Optional for Colab)

In [ ]:
# Mount Google Drive (only if running in Colab)
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = '/content/drive/MyDrive/JobScraper_Results'
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f"Google Drive mounted! Results will be saved to: {RESULTS_DIR}")
else:
    RESULTS_DIR = 'results'
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print(f"Running locally. Results will be saved to: {RESULTS_DIR}")

## 7. Configure Search Parameters (Optional Location Filter)

**MODIFY THIS CELL TO CHANGE YOUR SEARCH PARAMETERS**

Available job types:
- **PMjobs**: Product/Project Management positions
- **Datajobs**: Data Engineering & Analytics positions
- **AIjobs**: AI/ML positions
- **Autojobs**: Automotive Engineering positions
- **ResAsstjobs**: Research & Graduate Research positions
- **VLSIjobs**: VLSI/Chip Design Engineering positions
- **ECEjobs**: Electronics & Communications Engineering positions

**Location Filter (Optional):** Leave as `None` for global search, or provide a list of location strings (e.g., `["Hamburg", "Bremen"]`)

Set `JOB_TYPES` to a list of job types you want to scrape. The scraper will loop through each one.

In [ ]:
# SELECT YOUR PREFERENCES HERE
# Set multiple job types to scrape them all in sequence
JOB_TYPES = ["Datajobs", "PMjobs"]  # Modify as needed
# Available: AIjobs, PMjobs, Datajobs, Autojobs, ResAsstjobs, VLSIjobs, ECEjobs

# Optional: Filter by location(s). Leave as None for global search.
# Examples:
#   CUSTOM_LOCATIONS = None                        # Global search (default)
#   CUSTOM_LOCATIONS = ["Hamburg"]                 # German cities
#   CUSTOM_LOCATIONS = ["Milano", "Roma"]          # Italian cities
CUSTOM_LOCATIONS = None

# Employment type filters
INCLUDE_WORKING_STUDENT = True
INCLUDE_NON_WORKING_STUDENT = True

## 8. Run the Job Scraper (Loop Through All Selected Job Types)

⚠️ **NOTE:** This may take several minutes depending on the number of job types and search terms.
The scraper will process jobs one type at a time. Be patient and wait for completion.

In [ ]:
try:
    all_saved_files = []
    total_jobs_saved = 0
    
    # Loop through each job type
    for i, job_type in enumerate(JOB_TYPES, 1):
        print(f"\n{'#'*70}")
        print(f"# PROCESSING JOB TYPE {i}/{len(JOB_TYPES)}: {job_type}")
        print(f"{'#'*70}\n")
        
        try:
            # Get job class
            job_class = get_job_class(job_type)
            log_time(f"Starting scraper for {job_type}...")
            
            # Run scraper with optional location filter
            df_ws, df_nonws = process_and_save_results(job_class, INCLUDE_WORKING_STUDENT, INCLUDE_NON_WORKING_STUDENT, custom_locations=CUSTOM_LOCATIONS)
            
            # Save results
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            
            log_time(f"\n=== Saving Results for {job_type} ===")
            
            if INCLUDE_WORKING_STUDENT and df_ws is not None and len(df_ws) > 0:
                ws_file = f"{RESULTS_DIR}/{job_type}_workingstudent_{timestamp}.csv"
                df_ws.to_csv(ws_file, index=False)
                all_saved_files.append(ws_file)
                total_jobs_saved += len(df_ws)
                log_time(f"[OK] Saved {len(df_ws)} working student jobs to {ws_file}")
            
            if INCLUDE_NON_WORKING_STUDENT and df_nonws is not None and len(df_nonws) > 0:
                nonws_file = f"{RESULTS_DIR}/{job_type}_fulltime_{timestamp}.csv"
                df_nonws.to_csv(nonws_file, index=False)
                all_saved_files.append(nonws_file)
                total_jobs_saved += len(df_nonws)
                log_time(f"[OK] Saved {len(df_nonws)} non-working student jobs to {nonws_file}")
            
            if INCLUDE_WORKING_STUDENT or INCLUDE_NON_WORKING_STUDENT:
                if df_ws is None or len(df_ws) == 0:
                    if df_nonws is None or len(df_nonws) == 0:
                        log_time(f"No jobs found for {job_type}")
        
        except Exception as e:
            log_time(f"ERROR processing {job_type}: {str(e)}")
            import traceback
            traceback.print_exc()
    
    # Summary
    print(f"\n{'='*70}")
    print(f"ALL JOB TYPES PROCESSED")
    print(f"{'='*70}")
    print(f"Total files saved: {len(all_saved_files)}")
    print(f"Total jobs saved: {total_jobs_saved}")
    print(f"{'='*70}\n")
    
except Exception as e:
    log_time(f"CRITICAL ERROR: {str(e)}")
    import traceback
    traceback.print_exc()

## 9. Preview Results

In [ ]:
# Preview working student jobs
if INCLUDE_WORKING_STUDENT and df_ws is not None and len(df_ws) > 0:
    print(f"\n{'='*80}")
    print(f"WORKING STUDENT JOBS ({len(df_ws)} total)")
    print(f"{'='*80}")
    print(f"Columns: {list(df_ws.columns)}\n")
    with pd.option_context('display.max_columns', None, 'display.max_colwidth', 40):
        print(df_ws.head(10))
else:
    print("\nNo working student jobs found.")

# Preview non-working student jobs
if INCLUDE_NON_WORKING_STUDENT and df_nonws is not None and len(df_nonws) > 0:
    print(f"\n{'='*80}")
    print(f"NON-WORKING STUDENT JOBS ({len(df_nonws)} total)")
    print(f"{'='*80}")
    print(f"Columns: {list(df_nonws.columns)}\n")
    with pd.option_context('display.max_columns', None, 'display.max_colwidth', 40):
        print(df_nonws.head(10))
else:
    print("\nNo non-working student jobs found.")

## 10. View Saved Files

In [ ]:
# List all saved files (sorted by modification time - newest first)
if os.path.exists(RESULTS_DIR):
    saved = os.listdir(RESULTS_DIR)
    if saved:
        # Sort by modification time (newest first)
        saved.sort(key=lambda f: os.path.getmtime(os.path.join(RESULTS_DIR, f)), reverse=True)
    
    print(f"\n{'='*80}")
    print(f"SAVED FILES ({len(saved)} total) - Newest First")
    print(f"{'='*80}")
    if saved:
        for f in saved:
            file_path = os.path.join(RESULTS_DIR, f)
            size = os.path.getsize(file_path) / 1024  # Size in KB
            mod_time = datetime.fromtimestamp(os.path.getmtime(file_path)).strftime("%Y-%m-%d %H:%M:%S")
            print(f"  {f}")
            print(f"    Size: {size:.1f} KB | Modified: {mod_time}")
    else:
        print("  No files saved yet.")
    
    if IN_COLAB:
        print(f"\n📁 Files saved to: My Drive > JobScraper_Results")
    else:
        print(f"\n📁 Files saved to: {RESULTS_DIR}")

## 11. Documentation and Tips

### Quick Start Guide:
1. **Run cells 1-7 first** (setup, install dependencies, configuration)
2. **Modify cell 7** with your preferred job types and optional location filter
3. **Run cell 8** to execute the scraper (takes 5-15 minutes depending on volume)
4. **View results** in cells 9-10

### Available Job Types:
- **AIjobs**: Machine Learning, Artificial Intelligence, Data Science roles
- **PMjobs**: Product/Project Management roles
- **Datajobs**: Data Engineering & Analytics positions
- **Automobilejobs**: Automotive Engineering positions
- **ResearchAssistantjobs**: Research & Graduate Research positions
- **VLSIjobs**: VLSI/Chip Design Engineering positions
- **ECEjobs**: Electronics & Communications Engineering positions

### Location Filter Examples:
```python
# Global search (default) - searches worldwide
CUSTOM_LOCATIONS = None

# Filter for specific German cities (example with Hamburg)
CUSTOM_LOCATIONS = ["Hamburg"]

# Filter for Italian cities
CUSTOM_LOCATIONS = ["Milano", "Roma", "Torino"]

# Single city filter
CUSTOM_LOCATIONS = ["Hamburg"]

# Any custom location names
CUSTOM_LOCATIONS = ["London", "Berlin", "Amsterdam"]
```

### Output Columns (in order):
`site`, `title`, `company`, `location`, `date_posted`, `job_url`, `job_url_direct`, `salary_source`, `salary`, `job_type`, `is_remote`, and more...

### Features:
✅ Location-agnostic search (global by default)  
✅ Optional location filtering by custom cities/regions  
✅ Automatic language requirement filtering  
✅ Working student vs full-time job separation  
✅ Google Drive integration for easy access  
✅ Timestamped CSV files for version control  
✅ URLs formatted with trailing space for Excel compatibility  

### Customization:
- Modify `RESULTS_WANTED` (default: 300) to get more/fewer results
- Modify `HOURS_OLD` (default: 168 days = 7 days) to search different time periods
- Add custom keywords to `GERMAN_REQUIRED_KEYWORDS` or `STUDENT_KEYWORDS`
- Change `SITES` from `["linkedin", "google"]` to just one source if needed
- Provide `CUSTOM_LOCATIONS` to filter by specific cities/regions

### Troubleshooting:
- **"No jobs found"**: Try different job type or adjust date range
- **Connection errors**: The scraper may be rate-limited; wait a few minutes and retry
- **Google Drive not mounting**: Run cell 6 again or check permissions

---

**Created for Google Colab • Job Scraper v2.0**